<a href="https://colab.research.google.com/github/anushabershilla/Data-science-ML/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.head())
print(df.isnull().sum())
df['Age'].fillna(df['Age'].mean(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)
print(df.isnull().sum())
print(df['Sex'].unique())
print(df['Embarked'].unique())
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['IsAlone'] = 1
df.loc[df['FamilySize'] > 1, 'IsAlone'] = 0

print(df[['SibSp','Parch','FamilySize','IsAlone']].head())

In [ ]:
print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [ ]:
df['Age'].fillna(df['Age'].mean(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)

/tmp/ipykernel_5018/3906006821.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].mean(), inplace=True)
/tmp/ipykernel_5018/3906006821.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [ ]:
print(df['Sex'].unique())
print(df['Embarked'].unique())

['male' 'female']
['S' 'C' 'Q']


In [ ]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'])
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name  Sex   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris    0  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...    1  38.0      1      0   
2                             Heikkinen, Miss. Laina    1  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)    1  35.0      1      0   
4                           Allen, Mr. William Henry    0  35.0      0      0   

             Ticket     Fare  Embarked_C  Embarked_Q  Embarked_S  
0         A/5 21171   7.2500       False       False        True  
1          PC 17599  71.2833        True       False       False  
2  STON/O2. 3101282   7.9250       False       False        True  
3            113803  53.1000

In [ ]:
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# X and y
y = df['Survived']
X = df.drop(['Survived', 'PassengerId', 'Name', 'Ticket'], axis=1)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#pipeline

pipeline = Pipeline([
    ('classifier', RandomForestClassifier())
])
# GridSearch
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, None]
}

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(X_train, y_train)

# Best parameters
print("Best Params:", grid.best_params_)

# Best model
best_model=grid.best_estimator_

# Prediction
y_pred =best_model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# New predictions
new_df= pd.DataFrame([
    {
    'Pclass': 3,
    'Sex': 0,
    'Age': 28,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 7.25,
    'Embarked_C': 0,
    'Embarked_Q': 0,
    'Embarked_S': 1,
    'FamilySize': 1,
    'IsAlone': 1
    }
])

new_df1= pd.DataFrame([
    {
    'Pclass': 1,
    'Sex': 1,
    'Age': 25,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 80,
    'Embarked_C': 1,
    'Embarked_Q': 0,
    'Embarked_S': 0,
    'FamilySize': 1,
    'IsAlone': 1
    }
])

new_pred=best_model.predict(new_df)
print("Prediction 1:",new_pred)
print("Prediction 2:",best_model.predict(new_df1))

Best Params: {'classifier__max_depth': 5, 'classifier__n_estimators': 100}
Accuracy: 0.7985074626865671

Report:
               precision    recall  f1-score   support

           0       0.80      0.88      0.84       157
           1       0.80      0.68      0.74       111

    accuracy                           0.80       268
   macro avg       0.80      0.78      0.79       268
weighted avg       0.80      0.80      0.80       268

Confusion matrix:
 [[138  19]
 [ 35  76]]
Prediction 1: [0]
Prediction 2: [1]


In [ ]:

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['IsAlone'] = 1
df.loc[df['FamilySize'] > 1, 'IsAlone'] = 0

print(df[['SibSp','Parch','FamilySize','IsAlone']].head())

   SibSp  Parch  FamilySize  IsAlone
0      1      0           2        0
1      1      0           2        0
2      0      0           1        1
3      1      0           2        0
4      0      0           1        1


In [ ]:
importance = best_model.named_steps['classifier'].feature_importances_
imp_df=pd.DataFrame({'Feature':X.columns,'Importance':importance})
imp_df.sort_values(by='Importance',ascending=False)
print(imp_df)

       Feature  Importance
0       Pclass    0.125173
1          Sex    0.429021
2          Age    0.110944
3        SibSp    0.028118
4        Parch    0.028604
5         Fare    0.155546
6   Embarked_C    0.023822
7   Embarked_Q    0.009499
8   Embarked_S    0.021599
9   FamilySize    0.058263
10     IsAlone    0.009411
